# Imoprt tools

https://colab.research.google.com/drive/1ytWvgt8v-UoYphKX9-VZEipZuDMo1ygX#scrollTo=hreaJiVm5Ja8

In [1]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

# Load Datasets


In [3]:
#load data set
df = pd.read_csv("/content/domain_specific_chatbot_data.csv")

#display
df.head()

,query,response,intent,domain
0,What are the side effects of the COVID-19 vacc...,Common side effects of the COVID-19 vaccine in...,side effects inquiry,healthcare
1,How can I schedule an appointment with my doctor?,You can schedule an appointment by calling our...,appointment booking,healthcare
2,What should I do if I miss a dose of my medica...,"If you miss a dose, take it as soon as you rem...",medication inquiry,healthcare
3,How can I check my account balance?,You can check your balance by logging into you...,balance inquiry,finance
4,What is the interest rate for a personal loan?,The current interest rate for personal loans i...,loan inquiry,finance


In [4]:
df.shape

(3000, 4)

# Data Preprocessing

In [5]:
df['query'][0]

'What are the side effects of the COVID-19 vaccine?'

In [6]:
df['response'][0]

'Common side effects of the COVID-19 vaccine include soreness at the injection site, fever, and fatigue.'

In [7]:
df['domain'].value_counts()

,count
domain,
finance,1800
healthcare,1200


In [8]:
df['intent'].value_counts()

,count
intent,
side effects inquiry,300
appointment booking,300
medication inquiry,300
balance inquiry,300
loan inquiry,300
lost card reporting,300
flu symptoms inquiry,300
contact update,300
loan repayment adjustment,300


In [9]:
from sklearn.model_selection import train_test_split

#split thr data into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [10]:
train_df.shape, val_df.shape

((2400, 4), (600, 4))

In [11]:
train_data = train_df.reset_index(drop=True)
validation_data = val_df.reset_index(drop=True)

In [12]:
import re
def clean_text(text):
  text = re.sub(r'\r\n', ' ', text)
  text = re.sub(r'\s', ' ', text)
  text = re.sub(r'<.*?>', '', text)
  text = text.strip().lower()
  return text

#apply cleaning to dialogue and summary columns
train_data['query'] = train_data['query'].apply(clean_text)
train_data['response'] = train_data['response'].apply(clean_text)

validation_data['query'] = validation_data['query'].apply(clean_text)
validation_data['response'] = validation_data['response'].apply(clean_text)



In [13]:
train_data.head()

,query,response,intent,domain
0,what should i do if i miss a dose of my medica...,"if you miss a dose, take it as soon as you rem...",medication inquiry,healthcare
1,what are the side effects of the covid-19 vacc...,common side effects of the covid-19 vaccine in...,side effects inquiry,healthcare
2,what are the symptoms of flu?,"flu symptoms include fever, cough, sore throat...",flu symptoms inquiry,healthcare
3,how do i update my contact details on my account?,"to update your contact details, log into your ...",contact update,finance
4,what are the side effects of the covid-19 vacc...,common side effects of the covid-19 vaccine in...,side effects inquiry,healthcare


# Tokenization

In [14]:
tokenizer = T5Tokenizer.from_pretrained('t5-base')

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [15]:
def preprocess_function(examples):
  inputs = tokenizer(examples['query'], padding="max_length", truncate=True, max_length=250)
  outputs = tokenizer(examples['response'], padding="max_length", truncate=True, max_length=250)
  inputs["labels"] = outputs["input_ids"]
  return inputs

In [16]:
#apply the preprecessing
train_dataset = train_data.apply(preprocess_function, axis=1)
validation_dataset = validation_data.apply(preprocess_function, axis=1)

In [17]:
train_dataset[0]

{'input_ids': [125, 225, 3, 23, 103, 3, 99, 3, 23, 3041, 3, 9, 6742, 13, 82, 7757, 58, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

# Fine Tuning Model

In [18]:
model = T5ForConditionalGeneration.from_pretrained('t5-base')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# Cấu hình các tham số và thiết lập cho quá trình huấn luyện mô hình
training_args = TrainingArguments(
    output_dir="./results",  # Thư mục lưu các checkpoint và kết quả huấn luyện sau mỗi bước/epoch
    num_train_epochs=6,  # Giảm số epochs xuống 3 để chạy nhanh hơn trên Colab
    per_device_train_batch_size=8,  # Giảm batch size xuống 4 để tránh tràn bộ nhớ GPU (OOM)
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,  # Tần suất ghi log sau mỗi 10 bước
    save_steps=500,
    eval_steps=50,  # Đã sửa từ 'eval_step' thành 'eval_steps'
    eval_strategy="epoch",  # Thời điểm đánh giá: sau mỗi epoch
    report_to="none"  # Không đẩy log lên wandb
)

# Chuyển đổi Pandas Series chứa dicts thành một List của các dicts hợp lệ cho Trainer
train_list = train_dataset.tolist()
validation_list = validation_dataset.tolist()

# Khởi tạo đối tượng Trainer
trainer = Trainer(
    model=model,  # Mô hình T5
    args=training_args,  # Tham số huấn luyện đã sửa đổi
    train_dataset=train_list,  # Tập huấn luyện định dạng list
    eval_dataset=validation_list,  # Tập đánh giá định dạng list
)

# Bắt đầu chạy quá trình huấn luyện mô hình
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.032004,0.006436
2,0.001551,0.000036


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Save and load model

In [ ]:
model.save_pretrained("./chatbot_model")
tokenizer.save_pretrained("./chatbot_model")

In [ ]:
model = T5ForConditionGeneration.from_pretrained("./chatbot_model")
tokenizer = T5Tokenizer.from_pretrained("./chatbot_model")

# Chatbot system

In [ ]:
device = model.device

def chatbot(query):
  query = clean_text(query)
  input_ids = tokenizer(query, return_tensors="pt", max_length=250, truncation=True)

  inputs = {key: value.to(device) for key, value in inputs.items()}

  outputs = model.generate(
      input_ids=input_ids["input_ids"],
      max_length=250,
      num_beams=5,
      early_stopping=True
  )

  return tokenizer.decode(outputs[0], skip_special_tokens=True)

while True:
  user_input = input("You: ")
  if user_input.lower() == "quit":
    break
  response = chatbot(user_input)
  print("Chatbot: ", response)

In [ ]:
import shutil

#compress the saved model directory
shutil.make_archive("chatbot_model", "zip", "./chatbot_model")

from google.colab import files

#download the zipfile to pc
files.download("chatbot_model.zip")